# Run API Uncertainty Assessments Using Claude

This notebook loads the assessment templates created with `generate_all_assessments.py` and runs them through Claude API to get uncertainty assessments.

In [22]:
import os
import json
import glob
import time
import pandas as pd
import configparser
import re
from pathlib import Path
from anthropic import AnthropicBedrock
from botocore.config import Config

# Function to safely load AWS credentials from different sources
def load_aws_credentials():
    """Load AWS credentials from environment variables or AWS credentials file."""
    access_key = None
    secret_key = None
    
    # Try loading from environment variables directly
    access_key = os.environ.get("BEDROCK_ACCESS_KEY") or os.environ.get("AWS_ACCESS_KEY_ID")
    secret_key = os.environ.get("BEDROCK_SECRET_ACCESS_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY")
    
    # Try loading from .env file if python-dotenv is available
    if not access_key or not secret_key:
        try:
            from dotenv import load_dotenv
            # Look for .env file in the current directory
            load_dotenv()  
            access_key = os.environ.get("BEDROCK_ACCESS_KEY") or os.environ.get("AWS_ACCESS_KEY_ID")
            secret_key = os.environ.get("BEDROCK_SECRET_ACCESS_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY")
        except ImportError:
            print("python-dotenv not installed. Skipping .env file loading.")
    
    # If still not found, try AWS credentials file
    if not access_key or not secret_key:
        try:
            config = configparser.ConfigParser()
            config.read(os.path.expanduser("~/.aws/credentials"))
            if "default" in config:
                access_key = access_key or config["default"].get("aws_access_key_id")
                secret_key = secret_key or config["default"].get("aws_secret_access_key")
        except Exception as e:
            print(f"Could not read AWS credentials file: {str(e)}")
    
    return access_key, secret_key

# Load AWS credentials securely
BEDROCK_ACCESS_KEY, BEDROCK_SECRET_ACCESS_KEY = load_aws_credentials()

# Check if credentials are loaded or prompt for manual entry
if not BEDROCK_ACCESS_KEY or not BEDROCK_SECRET_ACCESS_KEY:
    print("AWS credentials not found in environment variables or config files.")
    print("You can enter them manually (not recommended for shared notebooks)")
    print("or use AWS credential provider chain for authentication.")
    print("")
    print("Recommended approaches:")
    print("1. Create a .env file in this directory with BEDROCK_ACCESS_KEY and BEDROCK_SECRET_ACCESS_KEY")
    print("2. Configure AWS credentials using 'aws configure' command")
    print("3. Set environment variables before launching Jupyter")

## Set up Claude Clients

In [23]:
def create_bedrock_client(region="us-west-2", max_retries=10000):
    """Create a Bedrock client with credentials if available, otherwise use AWS credential provider chain."""
    if BEDROCK_ACCESS_KEY and BEDROCK_SECRET_ACCESS_KEY:
        return AnthropicBedrock(
            aws_access_key=BEDROCK_ACCESS_KEY,
            aws_secret_key=BEDROCK_SECRET_ACCESS_KEY,
            aws_region=region,
            max_retries=max_retries
        )
    else:
        # Use default AWS credential provider chain
        return AnthropicBedrock(
            aws_region=region,
            max_retries=max_retries
        )

# Create main client with US West region
claude_client = create_bedrock_client(region="us-west-2")

# Create an alternative client with different region if needed
def initial_claude():
    return create_bedrock_client(region="us-east-1", max_retries=3000)

# Function to make predictions with Claude
def claude_pred(client, prompt):
    """Get a prediction from Claude API."""
    message = client.messages.create(
        model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # Change model as needed
        temperature=0,
        max_tokens=10060,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    prediction = message.content[0].text
    return prediction

## Find Assessment Files

In [36]:
# Base directory for the assessment templates
assessment_dir = "api_assessments"
# results_dir = "api_assessment_results_1"
results_dir = "api_assessment_results_2"

# Create the results directory if it doesn't exist
os.makedirs(results_dir, exist_ok=True)

# Find all assessment template files recursively
assessment_files = []
for root, dirs, files in os.walk(assessment_dir):
    for file in files:
        if file.endswith(".txt"):
            assessment_files.append(os.path.join(root, file))

print(f"Found {len(assessment_files)} assessment template files")

Found 774 assessment template files


## Assessment Function

In [37]:
def extract_numeric_values(text):
    """Extract numeric values from text, handling various formats including formulas."""
    # First, check if there's a direct formula with equals sign and extract the result
    # For example: "2 / (4 × 2) = 0.25"
    formula_match = re.search(r'=\s*(\d+\.?\d*)', text)
    if formula_match:
        return float(formula_match.group(1))
    
    # If there's a formula but no equals sign, try to parse the formula directly
    # Look for specific pattern like "2 / (4 × 2)" and calculate it
    div_pattern = re.search(r'(\d+)\s*/\s*\((\d+)\s*[×x*]\s*(\d+)\)', text)
    if div_pattern:
        try:
            numerator = int(div_pattern.group(1))
            denominator1 = int(div_pattern.group(2))
            denominator2 = int(div_pattern.group(3))
            return numerator / (denominator1 * denominator2)
        except Exception:
            pass
    
    # Check for number after colon (e.g., "Normalized Score: 0.25")
    colon_match = re.search(r':\s*(\d+\.?\d*)', text)
    if colon_match:
        return float(colon_match.group(1))
    
    # Last resort, look for any standalone number
    any_number_match = re.search(r'(\d+\.?\d*)', text)
    if any_number_match:
        return float(any_number_match.group(1))
    
    return None

def run_assessment(assessment_file, client):
    """Runs an assessment using the Claude API and saves the results."""
    # Read the assessment template
    with open(assessment_file, 'r') as f:
        prompt = f.read()
    
    # Get the relative parts of the file path
    rel_path = os.path.relpath(assessment_file, assessment_dir)
    parts = os.path.normpath(rel_path).split(os.sep)
    
    if len(parts) >= 3:
        environment = parts[0]
        function_name = parts[1]
        uncertainty_type = os.path.splitext(parts[2])[0]
    else:
        environment = "unknown"
        function_name = "unknown"
        uncertainty_type = "unknown"
    
    # Create directory for results if it doesn't exist
    result_dir = os.path.join(results_dir, environment, function_name)
    os.makedirs(result_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(result_dir, f"{uncertainty_type}_result.json")
    
    # Check if result already exists
    if os.path.exists(output_file):
        print(f"Result for {assessment_file} already exists. Skipping.")
        with open(output_file, 'r') as f:
            result = json.load(f)
        return result
    
    # Get prediction from Claude
    try:
        start_time = time.time()
        prediction = claude_pred(client, prompt)
        end_time = time.time()
        
        # Parse the prediction to extract the assessment scores
        total_score = None
        normalized_score = None
        likelihood = None
        
        # Try to find the overall assessment section and extract scores
        lines = prediction.split("\n")
        for i, line in enumerate(lines):
            if "Total Score:" in line:
                total_score = extract_numeric_values(line)
            
            if "Normalized Score:" in line:
                normalized_score = extract_numeric_values(line)
            
            if "Likelihood:" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    likelihood = parts[1].strip()
        
        # If we couldn't extract the normalized score directly, look for the normalization formula
        if normalized_score is None and total_score is not None:
            # Look for lines that contain the normalization formula
            for line in lines:
                if "Normalized Score" in line and ("/" in line) and any(x in line for x in ["×", "x", "*", "2"]):
                    # Try to extract the formula components: numerator / (denominator × 2)
                    div_pattern = re.search(r'(\d+)\s*/\s*\((\d+)\s*[×x*]\s*(\d+)\)', line)
                    if div_pattern:
                        try:
                            numerator = int(div_pattern.group(1))
                            denominator1 = int(div_pattern.group(2))
                            denominator2 = int(div_pattern.group(3))
                            normalized_score = numerator / (denominator1 * denominator2)
                        except Exception as e:
                            print(f"Error calculating normalized score: {str(e)}")
        
        # Store the result
        result = {
            "environment": environment,
            "function_name": function_name,
            "uncertainty_type": uncertainty_type,
            "prediction": prediction,
            "total_score": total_score,
            "normalized_score": normalized_score,
            "likelihood": likelihood,
            "execution_time": end_time - start_time,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        # Save the result
        with open(output_file, 'w') as f:
            json.dump(result, f, indent=2)
        
        return result
    
    except Exception as e:
        print(f"Error running assessment for {assessment_file}: {str(e)}")
        return None

## Run Batch Assessments

In [38]:
def run_batch_assessments(file_list, batch_size=10, max_files=None):
    """Run assessments in batches to avoid overwhelming the API."""
    if max_files is not None:
        file_list = file_list[:max_files]
    
    results = []
    
    for i in range(0, len(file_list), batch_size):
        batch_files = file_list[i:i+batch_size]
        print(f"\nProcessing batch {i//batch_size + 1}/{(len(file_list) + batch_size - 1)//batch_size}")
        
        for idx, file in enumerate(batch_files):
            print(f"Processing {idx+1}/{len(batch_files)}: {os.path.basename(file)}")
            result = run_assessment(file, claude_client)
            if result:
                results.append(result)
        
        if i + batch_size < len(file_list):
            print("Sleeping between batches...")
            time.sleep(5)
    
    return results

## Run Assessments with Multiprocessing

In [39]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_assessment_worker(assessment_file):
    """Worker function to run in a separate process"""
    # Create a new client for each worker to avoid sharing connections
    client = create_bedrock_client()
    return assessment_file, run_assessment(assessment_file, client)

def run_parallel_assessments(file_list, max_workers=None, max_files=None):
    """Run assessments in parallel using multiprocessing.
    
    Args:
        file_list: List of assessment files
        max_workers: Maximum number of parallel processes (None = CPU count)
        max_files: Maximum number of files to process (None for all)
        
    Returns:
        List of results
    """
    if max_files is not None:
        file_list = file_list[:max_files]
    
    if max_workers is None:
        max_workers = multiprocessing.cpu_count()
    
    results = []
    print(f"Starting parallel processing with {max_workers} workers...")
    total_files = len(file_list)
    
    # Use ProcessPoolExecutor to manage worker processes
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_file = {executor.submit(run_assessment_worker, file): file for file in file_list}
        
        # Process results as they complete without tqdm
        completed = 0
        print(f"Processing {total_files} assessments in parallel:")
        for future in as_completed(future_to_file):
            completed += 1
            # Print progress every 10 completions or when all are done
            if completed % 10 == 0 or completed == total_files:
                print(f"Progress: {completed}/{total_files} ({completed/total_files:.1%})")
            
            try:
                file_path, result = future.result()
                if result:
                    results.append(result)
            except Exception as exc:
                file = future_to_file[future]
                print(f"{file} generated an exception: {exc}")
    
    print(f"Completed processing {len(results)} assessments successfully")
    return results

In [40]:
# Run a sample assessment
if assessment_files:
    sample_file = assessment_files[0]
    print(f"Running sample assessment for {sample_file}")
    sample_result = run_assessment(sample_file, claude_client)

Running sample assessment for api_assessments/TimeNotificationEnv/delete_alarm/feature_limitation_error.txt


In [41]:
# NOTE: This will make API calls and may incur costs!
# Set max_files to a small number for testing
max_files_to_process = None # 10  # Set to None to process all files

# Choose which function to use:
# For sequential processing:
# results = run_batch_assessments(assessment_files, batch_size=5, max_files=max_files_to_process)

# For parallel processing:
results = run_parallel_assessments(assessment_files, max_workers=50, max_files=max_files_to_process)

Starting parallel processing with 50 workers...
Result for api_assessments/TimeNotificationEnv/delete_alarm/feature_limitation_error.txt already exists. Skipping.
Processing 774 assessments in parallel:
Progress: 10/774 (1.3%)
Progress: 20/774 (2.6%)
Progress: 30/774 (3.9%)
Progress: 40/774 (5.2%)
Progress: 50/774 (6.5%)
Progress: 60/774 (7.8%)
Progress: 70/774 (9.0%)
Progress: 80/774 (10.3%)
Progress: 90/774 (11.6%)
Progress: 100/774 (12.9%)
Progress: 110/774 (14.2%)
Progress: 120/774 (15.5%)
Progress: 130/774 (16.8%)
Progress: 140/774 (18.1%)
Progress: 150/774 (19.4%)
Progress: 160/774 (20.7%)
Progress: 170/774 (22.0%)
Progress: 180/774 (23.3%)
Progress: 190/774 (24.5%)
Progress: 200/774 (25.8%)
Progress: 210/774 (27.1%)
Progress: 220/774 (28.4%)
Progress: 230/774 (29.7%)
Progress: 240/774 (31.0%)
Progress: 250/774 (32.3%)
Progress: 260/774 (33.6%)
Progress: 270/774 (34.9%)
Progress: 280/774 (36.2%)
Progress: 290/774 (37.5%)
Progress: 300/774 (38.8%)
Progress: 310/774 (40.1%)
Progres

In [42]:
# Load assessment results for analysis
def load_results():
    all_results = []
    for root, dirs, files in os.walk(results_dir):
        for file in files:
            if file.endswith("_result.json"):
                try:
                    with open(os.path.join(root, file), 'r') as f:
                        result = json.load(f)
                        all_results.append(result)
                except Exception as e:
                    print(f"Error loading {os.path.join(root, file)}: {str(e)}")
    
    print(f"Loaded {len(all_results)} assessment results")
    return all_results

all_results = load_results()
df = pd.DataFrame(all_results)

Loaded 774 assessment results


In [43]:
# Display basic results summary
df.head()

,environment,function_name,uncertainty_type,prediction,total_score,normalized_score,likelihood,execution_time,timestamp
0,TimeNotificationEnv,delete_alarm,ambiguous_documentation,# Assessment of Ambiguous Documentation/Argume...,1.0,0.10,Low (0-0.33),15.585958,2025-06-18 01:49:51
1,TimeNotificationEnv,delete_alarm,feature_limitation_error,# Assessment of Feature Limitation Error Likel...,2.0,0.25,Low (0-0.33),18.721961,2025-06-18 01:49:35
2,TimeNotificationEnv,delete_alarm,system_failure_error,# Assessment of System Failure Error Likelihoo...,4.0,0.40,Moderate (0.34-0.66),17.131222,2025-06-18 01:49:53
3,TimeNotificationEnv,delete_alarm,completely_irrelevant_information,# Assessment of Completely Irrelevant Informat...,2.0,0.25,Low (0-0.33),17.131799,2025-06-18 01:49:53
4,TimeNotificationEnv,delete_alarm,unclear_functionality_boundaries,# Assessment of Unclear Functionality Boundari...,6.0,0.75,High (0.67-1.0),18.631612,2025-06-18 01:49:54
